In [9]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score

# Sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import VarianceThreshold

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


In [10]:
# Carregar o dataset DoS
dos_path = Path("../data/raw/MQTT Under Attack Dataset/DoS.csv")
df = pd.read_csv(dos_path)

print(f"📊 Shape do dataset: {df.shape}")
print(f"📝 Colunas: {df.shape[1]}")
print(f"📋 Registros: {df.shape[0]}")
print(f"\n🏷️ Distribuição do Label ('type'):")
print(df['type'].value_counts())

📊 Shape do dataset: (94625, 67)
📝 Colunas: 67
📋 Registros: 94625

🏷️ Distribuição do Label ('type'):
type
normal    49111
DoS       45514
Name: count, dtype: int64


In [11]:
# Criar coluna publish_gap com NaN para todos
df['publish_gap'] = np.nan

# Calcular o intervalo apenas para linhas PUBLISH (msgtype == 3)
publish_mask = df['mqtt.msgtype'] == 3
df.loc[publish_mask, 'publish_gap'] = df.loc[publish_mask, 'frame.time_epoch'].diff()

#Utilizando foward fill para propagar último valor conhecido
df['publish_gap'] = df['publish_gap'].ffill().fillna(0)


# Verificar resultado
print(f"Registros PUBLISH: {publish_mask.sum()}")
print(f"Valores não-nulos em publish_gap: {df['publish_gap'].notna().sum()}")
print(f"\nEstatísticas do publish_gap:")
print(df['publish_gap'].describe())

Registros PUBLISH: 37863
Valores não-nulos em publish_gap: 94625

Estatísticas do publish_gap:
count    94625.000000
mean         4.097958
std         10.853330
min          0.000000
25%          0.000000
50%          0.194238
75%          2.425834
max         57.168259
Name: publish_gap, dtype: float64


In [12]:
# Salvar o dataset com a nova feature publish_gap
output_path = Path("../data/processed/DoS_with_publish_gap.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print(f"✅ Dataset salvo em: {output_path}")
print(f"📊 Shape final: {df.shape}")
print(f"📝 Colunas: {list(df.columns)}")
print(f"\n🆕 Nova feature 'publish_gap' adicionada com sucesso!")

✅ Dataset salvo em: ../data/processed/DoS_with_publish_gap.csv
📊 Shape final: (94625, 68)
📝 Colunas: ['frame.time_delta', 'frame.time_delta_displayed', 'frame.time_epoch', 'frame.time_invalid', 'frame.time_relative', 'ip.src', 'ip.dst', 'tcp.srcport', 'tcp.dstport', 'eth.src', 'eth.dst', 'frame.cap_len', 'frame.coloring_rule.name', 'frame.coloring_rule.string', 'frame.comment', 'frame.comment.expert', 'frame.encap_type', 'frame.file_off', 'frame.ignored', 'frame.incomplete', 'frame.interface_id', 'frame.interface_name', 'frame.len', 'frame.link_nr', 'frame.marked', 'frame.md5_hash', 'frame.number', 'frame.offset_shift', 'mqtt.clientid', 'mqtt.clientid_len', 'mqtt.conack.flags', 'mqtt.conack.flags.reserved', 'mqtt.conack.flags.sp', 'mqtt.conack.val', 'mqtt.conflag.cleansess', 'mqtt.conflag.passwd', 'mqtt.conflag.qos', 'mqtt.conflag.reserved', 'mqtt.conflag.retain', 'mqtt.conflag.uname', 'mqtt.conflag.willflag', 'mqtt.conflags', 'mqtt.dupflag', 'mqtt.hdrflags', 'mqtt.kalive', 'mqtt.len',